In [1]:
import os
import json
import pandas as pd

BASE_DIR = os.path.abspath(
    os.path.join(os.getcwd(), ".." if os.path.basename(os.getcwd()) == "notebooks" else ".")
)

CONFIG_PATH = os.path.join(BASE_DIR, "notebooks", "config", "runtime_config.json")

if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError(f"runtime_config.json not found at {CONFIG_PATH}")

with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

INPUT_PATH = config["input_path"]

if not os.path.exists(INPUT_PATH):
    raise FileNotFoundError(f"Input file not found at {INPUT_PATH}")

# Output path (KEEP FIXED — do not make dynamic yet)
OUTPUT_FILE_PATH = "/media/ausaafkhan/New Volume1/project_exhibition/broken_dataset_repair/data/real/df_working.csv"

In [2]:
if not os.path.exists(INPUT_PATH):
    raise FileNotFoundError(f"[INGESTION ERROR] File not found at: {INPUT_PATH}")

In [3]:
df_original = pd.read_csv(INPUT_PATH)

In [4]:
# Ensure deterministic ID column exists
if "id" not in df_original.columns:
    print("No 'id' column found. Creating deterministic ID...")

    df_original.insert(0, "id", range(1, len(df_original) + 1))

No 'id' column found. Creating deterministic ID...


In [5]:
# Check if empty
if df_original.shape[0] == 0:
    raise ValueError("[INGESTION ERROR] Dataset has zero rows")

if df_original.shape[1] == 0:
    raise ValueError("[INGESTION ERROR] Dataset has zero columns")

# Check column names exist
if df_original.columns.isnull().any():
    raise ValueError("[INGESTION ERROR] Null column names detected")

# Check duplicate column names
if df_original.columns.duplicated().any():
    duplicates = df_original.columns[df_original.columns.duplicated()].tolist()
    raise ValueError(f"[INGESTION ERROR] Duplicate column names found: {duplicates}")

In [6]:
# Preserve order explicitly
df_original = df_original.loc[:, list(df_original.columns)]

# Create working copy
df_working = df_original.copy(deep=True)

In [7]:
dataset_profile = {
    "shape": df_original.shape,
    "columns": list(df_original.columns),
    "dtypes": df_original.dtypes.astype(str).to_dict()
}

In [8]:
output_dir = os.path.dirname(OUTPUT_FILE_PATH)

if not os.path.exists(output_dir):
    os.makedirs(output_dir, exist_ok=True)

In [9]:
df_working.to_csv(OUTPUT_FILE_PATH, index=False)

In [10]:
print("[INGESTION SUCCESS]")
print(f"Rows: {dataset_profile['shape'][0]}, Columns: {dataset_profile['shape'][1]}")
print(f"Saved to: {OUTPUT_FILE_PATH}")

[INGESTION SUCCESS]
Rows: 48842, Columns: 16
Saved to: /media/ausaafkhan/New Volume1/project_exhibition/broken_dataset_repair/data/real/df_working.csv
